## Create summary statistics for the clusters

In [1]:
import numpy as np
import pandas as pd
pd.options.mode.copy_on_write = True

In [ ]:
# From add_annot_to_clusters.ipynb
pdb_clusters = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters_annotated.tsv", sep="\t")
pdb_clusters["chain_id_0"] = pdb_clusters["chain_id_0"].fillna("NA")
pdb_clusters["chain_id_1"] = pdb_clusters["chain_id_1"].fillna("NA")

/var/folders/8n/b4ym5rbn48v7d2lxw_5ph7z40000gp/T/ipykernel_10019/2822720730.py:1: DtypeWarning: Columns (32,34) have mixed types. Specify dtype option on import or set low_memory=False.
  pdb_clusters = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_clusters_annotated.tsv", sep="\t")


## Cluster size & date

In [3]:
# Cluster size
cluster_size = pdb_clusters.groupby("intcluster").size()
cluster_summary = cluster_size.to_frame()
cluster_summary.columns = ["cluster_size"]

# Earliest date of deposition
pdb_ids_cluster_first_last = pd.read_csv("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/pdb_ids_cluster_first_last.tsv", header=None, index_col=False, sep="\t", names=["cluster_id","first_date","last_date","pdb_id_list"], dtype={"cluster_id":'Int64', "first_date":'Int64', "last_date":'Int64', "pdb_id_list":object})
cluster_summary = pd.merge(cluster_summary, pdb_ids_cluster_first_last[["cluster_id","first_date"]], left_index=True, right_on="cluster_id", how="left")
cluster_summary.drop("cluster_id", axis=1, inplace=True)

## Cluster TM-score

In [4]:
tm_scores = pd.read_csv("/Volumes/imb-luckgr/imb-luckgr2/projects/interface_clustering/interfaceclu40_0_qtm_ttm_u_t.tsv", sep="\t", header=None)
tm_scores.columns = ["query","target","qtm","ttm","u","t"]
tm_scores["query_id"] = [int(x.split("_")[0].split("DI")[1]) for x in tm_scores["query"]]
tm_scores["target_id"] = [int(x.split("_")[0].split("DI")[1]) for x in tm_scores["target"]]

tm_scores["intcluster"] = tm_scores["query_id"].map(dict(zip(pdb_clusters.old_complex_id, pdb_clusters.intcluster)))
cluster_qtms = tm_scores.groupby("intcluster")["qtm"].mean().to_frame()
cluster_qtms.columns = ["cluster_qtm"]

cluster_summary["cluster_qtm"] = cluster_qtms

## Taxonomy

In [5]:
# Intraspecies
pdb_clusters.loc[(pdb_clusters['tax_id_0'] == pdb_clusters['tax_id_1']), 'intraspecies'] = 1
pdb_clusters.loc[(pdb_clusters['tax_id_0'] != pdb_clusters['tax_id_1']), 'intraspecies'] = 0
pdb_clusters.loc[((pdb_clusters['tax_id_0'].isna()) | (pdb_clusters['tax_id_1'].isna())), "intraspecies"] = 0

intra_frac = pdb_clusters.groupby("intcluster")["intraspecies"].mean().to_frame()
intra_frac.columns = ["pct_intraspecies"]

In [6]:
# Number of species
def generate_tax_id_list_len(x):
    tax_ids = list(set(x["tax_id_0"]) | set(x["tax_id_1"]))
    return len([x for x in tax_ids if x >= 0])

num_species = pdb_clusters.groupby("intcluster").apply(generate_tax_id_list_len, include_groups=False)
num_species = num_species.to_frame()
num_species.columns = ["num_species"]

In [7]:
# Data merge
cluster_summary = pd.merge(cluster_summary, intra_frac, right_index=True, left_index=True, how="left")
cluster_summary = pd.merge(cluster_summary, num_species, right_index=True, left_index=True, how="left")

## Categorical variables
Prot-pep status, Coiled-coil, Pfam annotation, CATH annotation, Disorder, Antibody, GO terms

In [8]:
# Protein-peptide status
protpep_status = pdb_clusters.groupby("intcluster")["protpep_status"].value_counts().to_frame()
protpep_status.reset_index(inplace=True)
protpep_status = pd.pivot(protpep_status, index="intcluster", columns="protpep_status")
protpep_status.columns = ["pep-pep","prot-pep","prot-prot"]
protpep_status

top_protpep = pdb_clusters.groupby("intcluster")["protpep_status"].apply(lambda x: x.value_counts().idxmax()).to_frame()
top_protpep.columns = ["top_protpepcat"]

cluster_summary = pd.merge(cluster_summary, protpep_status, left_index=True, right_index=True, how="left")
cluster_summary["pct_protprot"] = cluster_summary["prot-prot"] / cluster_summary["cluster_size"]
cluster_summary["pct_protpep"] = cluster_summary["prot-pep"] / cluster_summary["cluster_size"]
cluster_summary["pct_peppep"] = cluster_summary["pep-pep"] / cluster_summary["cluster_size"]
cluster_summary = pd.merge(cluster_summary, top_protpep, left_index=True, right_index=True, how="left")
cluster_summary.drop(["prot-prot","prot-pep","pep-pep"], axis=1, inplace=True)

In [9]:
# Disorder categories
disorder = pdb_clusters.groupby("intcluster")["if_type"].value_counts().to_frame()
disorder.reset_index(inplace=True)
disorder = pd.pivot(disorder, index="intcluster", columns="if_type")
disorder.columns = ["dis-dis","dis-ord","ord-ord","un"]
disorder

top_discat = pdb_clusters.groupby("intcluster")["if_type"].apply(lambda x: x.value_counts().idxmax()).to_frame()
top_discat.columns = ["top_discat"]

cluster_summary = pd.merge(cluster_summary, disorder, left_index=True, right_index=True, how="left")
cluster_summary["pct_disdis"] = cluster_summary["dis-dis"] / cluster_summary["cluster_size"]
cluster_summary["pct_disord"] = cluster_summary["dis-ord"] / cluster_summary["cluster_size"]
cluster_summary["pct_ordord"] = cluster_summary["ord-ord"] / cluster_summary["cluster_size"]
cluster_summary = pd.merge(cluster_summary, top_discat, left_index=True, right_index=True, how="left")
cluster_summary.drop(["dis-dis","dis-ord","ord-ord","un"], axis=1, inplace=True)

In [10]:
# Coiled-coils
pdb_clusters.loc[pdb_clusters["coil_status"] == "Coiled-coil", "coil_status_int"] = 1
pdb_clusters.loc[pdb_clusters["coil_status"] != "Coiled-coil", "coil_status_int"] = 0

coils = pdb_clusters.groupby("intcluster")["coil_status_int"].mean().to_frame()
coils.columns = ["pct_coiledcoil"]

cluster_summary = pd.merge(cluster_summary, coils, left_index=True, right_index=True, how="left")

In [11]:
# Antibodies
pdb_clusters["antibody_int"] = 1
pdb_clusters.loc[pdb_clusters["antibody"].isna(), "antibody_int"] = 0
pdb_clusters.loc[pdb_clusters["antibody"] == "Antigen-Antigen", "antibody_int"] = 0
pdb_clusters.loc[pdb_clusters["antibody"] == "Antigen-Other", "antibody_int"] = 0

antibodies = pdb_clusters.groupby("intcluster")["antibody_int"].mean().to_frame()
antibodies.columns = ["pdb_ab"]

cluster_summary = pd.merge(cluster_summary, antibodies, left_index=True, right_index=True, how="left")

In [12]:
# Pfam annotations
pfams = pdb_clusters.groupby("intcluster")["pfam_annot"].describe()
pfams.columns = ["count","num_pfams","top_pfam","num_top_pfam"]
cluster_summary = pd.merge(cluster_summary, pfams, left_index=True, right_index=True, how="left")
cluster_summary["pct_top_pfam"] = cluster_summary["num_top_pfam"] / cluster_summary["cluster_size"]

# CATH annotations
caths = pdb_clusters.groupby("intcluster")["cath_annot"].describe()
caths.columns = ["count","num_caths","top_cath","num_top_cath"]
cluster_summary = pd.merge(cluster_summary, caths, left_index=True, right_index=True, how="left")
cluster_summary["pct_top_cath"] = cluster_summary["num_top_cath"] / cluster_summary["cluster_size"]

cluster_summary.drop(["count_x","count_y","num_top_pfam","num_top_cath"], axis=1, inplace=True)

In [13]:
# GO terms
go_only = pdb_clusters[["intcluster","go_cat"]]
go_only.dropna(subset=["go_cat"], axis=0, inplace=True)
go_only["go_cat"] = go_only.apply(lambda x: x["go_cat"].split(","), axis=1)
go_only = go_only.explode("go_cat")

go_cats = go_only.groupby("intcluster")["go_cat"].describe()
go_cats.columns = ["count","num_gos","top_go","num_top_go"]
cluster_summary = pd.merge(cluster_summary, go_cats, left_index=True, right_index=True, how="left")
cluster_summary["pct_top_go"] = cluster_summary["num_top_go"] / cluster_summary["cluster_size"]
cluster_summary.drop("num_top_go", axis=1, inplace=True)

## Interface size

In [14]:
pdb_clusters["if_size"] = pdb_clusters["maxiflen"] + pdb_clusters["miniflen"]
ifsize = pdb_clusters.groupby("intcluster")["if_size"].describe()
cluster_summary = pd.merge(cluster_summary, ifsize[["50%","min","max"]], left_index=True, right_index=True, how="left")
cluster_summary.rename(columns={"50%":"median_ifsize","max":"max_ifsize","min":"min_ifsize"}, inplace=True)

## Secondary structure at interface

In [15]:
median_fracs = pdb_clusters.groupby("intcluster")[["helix_frac","beta_frac","bendturn_frac","unassign_frac"]].median()
median_fracs.columns = ["median_helixfrac","median_betafrac","median_bendturnfrac","median_unassignfrac"]
cluster_summary = pd.merge(cluster_summary, median_fracs, left_index=True, right_index=True, how="left")

In [24]:
missing_dssp_cluids = list(set(cluster_summary[cluster_summary["median_helixfrac"].isna()].index))
print(len(missing_dssp_cluids))

1266


In [ ]:
dssp_na_pdb_ids = list(set(pdb_clusters[pdb_clusters["intcluster"].isin(missing_dssp_cluids)]["pdb_id"]))
dssp_na_pdb_ids = [x.split("-")[0] for x in dssp_na_pdb_ids]

with open("/Volumes/imb-luckgr/projects/interface_clustering/scripts/cluster_analysis/slurm_python_get-dssp.log", "r") as f:
    log_contents = f.readlines()

log_pdbids = [x[0:4] for x in log_contents[5:]]

# Check reasons for missing data - issues with source data, not with processing
dssp_na_notinlog = [x for x in dssp_na_pdb_ids if x not in log_pdbids]
print(len(dssp_na_notinlog))
print(dssp_na_notinlog)

918
['1l7z', '7x5q', '1xrp', '4u4r', '3ah8', '3egh', '6b0u', '7xeb', '3sjv', '8bqs', '5l9t', '7fgr', '7oyr', '7uqj', '1t3e', '5dc7', '6giq', '6ip4', '8on7', '8otz', '8t9a', '6ap1', '8hfr', '1ef1', '4pl8', '7z6a', '7pul', '5jr6', '5vf3', '6hu9', '5khu', '2a6k', '5juo', '4u4q', '8gym', '1w72', '7u0x', '1bcs', '7dgq', '7w5z', '4u50', '8fl8', '1n0w', '8jut', '8dk3', '1w72', '4jna', '5y28', '8k0g', '5dc6', '8wq2', '7uo8', '2mc0', '8ryp', '5ndw', '6syf', '8a3t', '6fbt', '1m21', '1vdm', '4hp2', '4v5o', '6xyw', '8ryo', '9e5c', '7wtw', '5dat', '3mgb', '6azm', '9f5p', '1by5', '2ce9', '2l6j', '3u51', '4wkm', '1mi5', '6im4', '1upk', '5fci', '4u3u', '8j9i', '4v8o', '5mei', '6ef3', '7arl', '5fjx', '7uma', '7z5y', '1j4k', '6yaf', '4u52', '6ab6', '6im4', '6z3m', '1xqy', '7y7a', '7klu', '2ilm', '3f2k', '3zg5', '6hk3', '4v5n', '7wae', '9dlz', '6t15', '1k2m', '5b5n', '7yqk', '8v41', '7y04', '2m9q', '1nx0', '6pp6', '1jg3', '8v9k', '8yvz', '4u4y', '4nec', '8j9h', '7o4k', '5xof', '5nd8', '6pos', '6c23', '9c

## PDB characteristics

In [29]:
keywords = pdb_clusters.groupby("intcluster")["pdb_keyword"].describe()
keywords.columns = ["count","num_keywords","top_keyword","num_top_keyword"]
cluster_summary = pd.merge(cluster_summary, keywords, left_index=True, right_index=True, how="left")
cluster_summary["pct_top_keyword"] = cluster_summary["num_top_keyword"] / cluster_summary["cluster_size"]

expts = pdb_clusters.groupby("intcluster")["pdb_expt_method"].describe()
expts.columns = ["count","num_exptmethods","top_exptmethod","num_top_exptmethod"]
cluster_summary = pd.merge(cluster_summary, expts, left_index=True, right_index=True, how="left")
cluster_summary["pct_top_exptmethod"] = cluster_summary["num_top_exptmethod"] / cluster_summary["cluster_size"]

cluster_summary.drop(["count_x","count_y","num_top_keyword","num_top_exptmethod"], axis=1, inplace=True)

In [30]:
median_res = pdb_clusters.groupby("intcluster")["pdb_resolution"].median().to_frame()
median_res.columns = ["median_resolution"]
cluster_summary = pd.merge(cluster_summary, median_res, left_index=True, right_index=True, how="left")

## Save dataframe

In [19]:
cluster_summary["intcluster"] = cluster_summary.index.map(dict(zip(pdb_clusters.intcluster, pdb_clusters.intcluster)))
cluster_summary

,cluster_size,first_date,cluster_qtm,pct_intraspecies,num_species,pct_protprot,pct_protpep,pct_peppep,top_protpepcat,pct_disdis,...,median_unassignfrac,num_keywords,top_keyword,pct_top_keyword,count,num_exptmethods,top_exptmethod,pct_top_exptmethod,median_resolution,intcluster
0,72,20130227,1.000000,1.000000,1,1.0,NaN,NaN,Protein-protein,NaN,...,0.1560,1,ISOMERASE,1.0,72,1,X-RAY DIFFRACTION,1.0,2.800,0
1,12,20120327,0.561286,0.333333,4,1.0,NaN,NaN,Protein-protein,NaN,...,0.0690,2,IMMUNE SYSTEM,0.666667,12,2,X-RAY DIFFRACTION,0.583333,3.085,1
2,2,20240327,1.000000,0.000000,0,1.0,NaN,NaN,Protein-protein,NaN,...,0.1920,1,VIRAL PROTEIN,1.0,2,1,X-RAY DIFFRACTION,1.0,1.800,2
3,2,20240327,1.000000,0.000000,0,1.0,NaN,NaN,Protein-protein,NaN,...,0.2155,1,VIRAL PROTEIN,1.0,2,1,X-RAY DIFFRACTION,1.0,1.800,3
4,1,20240327,1.000000,0.000000,0,1.0,NaN,NaN,Protein-protein,NaN,...,0.0470,1,VIRAL PROTEIN,1.0,1,1,X-RAY DIFFRACTION,1.0,1.800,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77162,2,20201211,1.000000,0.000000,3,1.0,NaN,NaN,Protein-protein,NaN,...,0.3325,1,TRANSCRIPTION,1.0,2,1,ELECTRON MICROSCOPY,1.0,2.790,77162
77163,3,20190625,0.943667,1.000000,1,1.0,NaN,NaN,Protein-protein,NaN,...,0.2220,2,MEMBRANE PROTEIN,0.666667,3,1,ELECTRON MICROSCOPY,1.0,3.700,77163
77164,1,20220929,1.000000,1.000000,1,1.0,NaN,NaN,Protein-protein,NaN,...,0.2520,1,METAL BINDING PROTEIN,1.0,1,1,X-RAY DIFFRACTION,1.0,4.600,77164
77165,2,20220930,1.000000,1.000000,1,1.0,NaN,NaN,Protein-protein,NaN,...,0.3000,1,SIGNALING PROTEIN,1.0,2,1,X-RAY DIFFRACTION,1.0,1.950,77165


In [20]:
cluster_summary.to_csv("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_cluster_summary.tsv", sep="\t", index=None, float_format="%.3f")